In [ ]:
from pathlib import Path
import os

def _project_root_from_env(variable, project_dir_name):
    candidates = [Path.cwd() / ".env", Path.cwd() / project_dir_name / ".env", Path.cwd().parent / ".env"]
    for env_path in candidates:
        if not env_path.is_file():
            continue
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            key, separator, value = line.strip().partition("=")
            if separator and key and not key.startswith("#"):
                os.environ.setdefault(key, value.strip().strip(chr(34)).strip(chr(39)))
    configured = os.environ.get(variable)
    if not configured:
        raise RuntimeError(f"Set {variable} in the project .env file or process environment.")
    root = Path(configured).expanduser().resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Configured project root does not exist: {root}")
    return root

PROJECT_ROOT = _project_root_from_env("TRUSTEDSQL_PROJECT_ROOT", "method_giang")
DDL_PATH = PROJECT_ROOT / "resources" / "schema" / "ddl.md"
OUTPUT_PATH = PROJECT_ROOT / "artifacts" / "schema" / "compact" / "v1" / "compact_schema.txt"
DATABASE_URL = os.environ.get("TRUSTEDSQL_DATABASE_URL") or os.environ.get("DATABASE_URL")

PROJECT_ROOT, DDL_PATH, OUTPUT_PATH, bool(DATABASE_URL)

In [ ]:
import sys

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from trustedsql.preprocessing.compact_schema import generate_compact_schema_prompt

result = generate_compact_schema_prompt(
    ddl_path=DDL_PATH,
    output_path=OUTPUT_PATH,
    database_url=DATABASE_URL,
    example_limit=3,
)

print(
    f"{result.table_count} tables, "
    f"{result.column_count} columns, "
    f"{result.relationship_count} relationships, "
    f"{result.example_column_count} example columns, "
    f"{result.characters} chars -> {result.output_path}"
)